In [2]:
import pandas as pd
import numpy as np

from src.testing_validation.model_test import calculate_rmse
from src.helpers import fit_cv_timeseries_model

from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

All Data

In [3]:

combined_data = (
    pd.read_parquet("../data/train.parquet")
    .sort_values("date")
    .reset_index(drop=True)
)

combined_data["usd_zar_28_movement"] = (
    combined_data["usd_zar_28"] - combined_data["usd_zar"]
)
combined_data.head(1)

,date,gold_usd_per_oz,platinum_usd_per_oz,gold_usd_per_oz_return,platinum_usd_per_oz_return,us_fed_funds,us_5y_yield,vix,broad_usd_index,iron_ore_usd_per_tonne,...,usd_zar,usd_zar_28,usd_zar_1w_return,usd_zar_1m_return,usd_zar_3m_return,usd_zar_1m_volatility,richards_bay_coal_usd,sa_5y_cds_bp,sa_5y_yield,usd_zar_28_movement
0,2008-10-10,855.400024,996.700012,-0.019289,-0.007625,0.79,2.77,69.95,97.999,60.8,...,9.3626,10.0284,0.085265,0.066904,-0.004945,0.043376,112.4,455.4,9.115,0.6658


In [4]:
X_all = combined_data.drop(columns=["date", "usd_zar", "usd_zar_28_movement"])

In [5]:
y = combined_data["usd_zar_28_movement"]

In [6]:
model = make_pipeline(
    StandardScaler(),
    MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42),
)
fit_cv_timeseries_model(model, X_all, y)

Fold 0, rmse: 0.47103357193394374
Fold 1, rmse: 1.0898650581005653
Fold 2, rmse: 0.8577021538296676
Fold 3, rmse: 0.7608703933667474
Fold 4, rmse: 1.25671002973969
RMSEs over all splits: [0.47103357193394374, 1.0898650581005653, 0.8577021538296676, 0.7608703933667474, 1.25671002973969]
Mean RMSE over all folds: 0.8872362413941228


np.float64(0.8872362413941228)

Engineered data

In [7]:

engineered_data = combined_data.copy().drop(columns=["date", "usd_zar_28", "usd_zar_28_movement"])
engineered_data["interest_rate_diff"] = (
    engineered_data["sa_repo_rate"] - engineered_data["us_fed_funds"]
)
engineered_data["sa_us_5y_yield_spread"] = (
    engineered_data["sa_5y_yield"] - engineered_data["us_5y_yield"]
)

engineered_data["commodities"] = np.mean([engineered_data.iron_ore_usd_per_tonne, 
engineered_data.gold_usd_per_oz, engineered_data.platinum_usd_per_oz, engineered_data.richards_bay_coal_usd], axis=0)

engineered_features = engineered_data.columns
# [
#     #"commodities",
#     #"usd_zar",
#     #"gold_usd_per_oz",
#     #"platinum_usd_per_oz",
#     #"richards_bay_coal_usd",
#     #"iron_ore_usd_per_tonne",
#     "brent_usd_per_barrel",
#     "interest_rate_diff",
#     "sa_us_5y_yield_spread",
#     "sa_yoy_inflation",
#     "sa_5y_cds_bp",
#     "vix",
#     "broad_usd_index",
#     "sa_cpi",
#     #"gold_usd_per_oz_return",
#     #"platinum_usd_per_oz_return",
#     #"usd_zar_1w_return",
#     #"usd_zar_1m_return",
#     #"usd_zar_3m_return",
#     #"usd_zar_1m_volatility",
# ]

X_engineered = engineered_data[engineered_features]
assert X_engineered.select_dtypes(exclude="number").empty

In [8]:
model = make_pipeline(
    StandardScaler(),
    MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42),
)
fit_cv_timeseries_model(model, X_engineered, y)

Fold 0, rmse: 0.8327099183649406
Fold 1, rmse: 1.1134907568253607
Fold 2, rmse: 1.3587994495684992
Fold 3, rmse: 1.8598862109260943
Fold 4, rmse: 1.812751103912617
RMSEs over all splits: [0.8327099183649406, 1.1134907568253607, 1.3587994495684992, 1.8598862109260943, 1.812751103912617]
Mean RMSE over all folds: 1.395527487919502


np.float64(1.395527487919502)

Backward stepwise feature selection

In [12]:
import time
import warnings

from joblib import Parallel, delayed, parallel_config
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import TimeSeriesSplit


def _progress(message):
    """Print immediately in Jupyter instead of waiting for a logging buffer."""
    print(f"{time.strftime('%H:%M:%S')} | {message}", flush=True)


def exhaustive_backward_stepwise_selection(
    estimator,
    X,
    y,
    n_splits=3,
    n_jobs=4,
    log_every=None,
):
    """Remove every feature, scoring all candidate removals at each step.

    The procedure does not stop when RMSE worsens. It evaluates the complete
    elimination path through the zero-feature fold-mean baseline, then returns
    the subset with the lowest CV RMSE encountered anywhere on that path.
    """
    selected_features = list(X.columns)
    splits = list(TimeSeriesSplit(n_splits=n_splits).split(X))
    total_start = time.perf_counter()
    n_jobs = max(1, min(n_jobs, len(selected_features)))
    base_estimator = clone(estimator)

    def cv_rmse(features, verbose=False):
        fold_rmses = []
        for fold, (train_idx, test_idx) in enumerate(splits, start=1):
            if features:
                fold_model = clone(base_estimator)
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", category=ConvergenceWarning)
                    fold_model.fit(
                        X.iloc[train_idx][features],
                        y.iloc[train_idx],
                    )
                predictions = fold_model.predict(X.iloc[test_idx][features])
            else:
                # A model cannot fit an empty matrix. The honest zero-feature
                # comparator predicts the training-fold mean.
                predictions = np.full(len(test_idx), y.iloc[train_idx].mean())
            fold_rmses.append(root_mean_squared_error(y.iloc[test_idx], predictions))
            if verbose:
                _progress(
                    f"Baseline fold {fold}/{len(splits)} complete; "
                    f"RMSE={fold_rmses[-1]:.6f}"
                )
        return float(np.mean(fold_rmses))

    def evaluate_candidate(feature, remaining):
        return cv_rmse(remaining), feature

    _progress(
        f"Starting exhaustive MLP elimination with {len(selected_features)} "
        f"features, {n_splits} folds and {n_jobs} candidate workers"
    )
    _progress("Fitting baseline CV now...")
    current_rmse = cv_rmse(selected_features, verbose=True)
    best_rmse = current_rmse
    best_features = selected_features.copy()
    history = [{
        "step": 0,
        "removed": None,
        "n_features": len(selected_features),
        "remaining_features": tuple(selected_features),
        "cv_rmse": current_rmse,
        "change_from_previous": np.nan,
        "best_so_far": True,
        "round_seconds": 0.0,
        "elapsed_seconds": time.perf_counter() - total_start,
    }]
    _progress(f"Baseline complete; mean CV RMSE={current_rmse:.6f}")

    step = 0
    while selected_features:
        step += 1
        round_start = time.perf_counter()
        candidates = [
            (feature, [name for name in selected_features if name != feature])
            for feature in selected_features
        ]
        report_every = log_every or max(1, len(candidates) // 5)
        _progress(
            f"Round {step}: evaluating {len(candidates)} removals "
            f"from {len(selected_features)} features"
        )

        completed_results = []
        with parallel_config(
            backend="loky",
            n_jobs=n_jobs,
            inner_max_num_threads=1,
        ):
            result_generator = Parallel(return_as="generator_unordered")(
                delayed(evaluate_candidate)(feature, remaining)
                for feature, remaining in candidates
            )
            for completed, result in enumerate(result_generator, start=1):
                completed_results.append(result)
                if completed % report_every == 0 or completed == len(candidates):
                    _progress(
                        f"Round {step}: {completed}/{len(candidates)} candidates "
                        f"complete ({time.perf_counter() - round_start:.1f}s)"
                    )

        candidate_rmse, feature_to_remove = min(completed_results)
        previous_rmse = current_rmse
        selected_features.remove(feature_to_remove)
        current_rmse = candidate_rmse
        is_best = current_rmse < best_rmse
        if is_best:
            best_rmse = current_rmse
            best_features = selected_features.copy()

        history.append({
            "step": step,
            "removed": feature_to_remove,
            "n_features": len(selected_features),
            "remaining_features": tuple(selected_features),
            "cv_rmse": current_rmse,
            "change_from_previous": current_rmse - previous_rmse,
            "best_so_far": is_best,
            "round_seconds": time.perf_counter() - round_start,
            "elapsed_seconds": time.perf_counter() - total_start,
        })
        _progress(
            f"Round {step} complete: removed '{feature_to_remove}'; "
            f"{len(selected_features)} remain; RMSE={current_rmse:.6f}"
        )

    _progress(
        f"Complete path finished in {time.perf_counter() - total_start:.1f}s; "
        f"best subset has {len(best_features)} features and RMSE={best_rmse:.6f}"
    )
    return best_features, pd.DataFrame(history)

In [13]:
selection_model = make_pipeline(
    StandardScaler(),
    MLPRegressor(
        hidden_layer_sizes=(32, 16),
        alpha=0.001,
        batch_size=128,
        max_iter=250,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=15,
        random_state=42,
    ),
)

print("Starting exhaustive MLP elimination...", flush=True)
best_features, selection_history = exhaustive_backward_stepwise_selection(
    selection_model,
    X_engineered,
    y,
    n_splits=3,
    n_jobs=4,
)

best_row = selection_history.loc[selection_history["cv_rmse"].idxmin()]
print(
    f"Best subset: {len(best_features)} of {X_engineered.shape[1]} features; "
    f"CV RMSE={best_row['cv_rmse']:.6f}",
    flush=True,
)
print(best_features)
selection_history

Starting exhaustive MLP elimination...
17:20:23 | Starting exhaustive MLP elimination with 25 features, 3 folds and 4 candidate workers
17:20:23 | Fitting baseline CV now...
17:20:23 | Baseline fold 1/3 complete; RMSE=0.986981
17:20:23 | Baseline fold 2/3 complete; RMSE=1.118915
17:20:24 | Baseline fold 3/3 complete; RMSE=1.793278
17:20:24 | Baseline complete; mean CV RMSE=1.299724
17:20:24 | Round 1: evaluating 25 removals from 25 features
17:20:28 | Round 1: 5/25 candidates complete (3.9s)
17:20:30 | Round 1: 10/25 candidates complete (5.6s)
17:20:32 | Round 1: 15/25 candidates complete (8.2s)
17:20:34 | Round 1: 20/25 candidates complete (10.0s)
17:20:36 | Round 1: 25/25 candidates complete (12.0s)
17:20:36 | Round 1 complete: removed 'iron_ore_usd_per_tonne'; 24 remain; RMSE=1.093896
17:20:36 | Round 2: evaluating 24 removals from 24 features
17:20:38 | Round 2: 4/24 candidates complete (2.0s)
17:20:40 | Round 2: 8/24 candidates complete (3.7s)
17:20:41 | Round 2: 12/24 candidates 

,step,removed,n_features,remaining_features,cv_rmse,change_from_previous,best_so_far,round_seconds,elapsed_seconds
0,0,NaN,25,"(gold_usd_per_oz, platinum_usd_per_oz, gold_us...",1.299724,NaN,True,0.000000,1.324398
1,1,iron_ore_usd_per_tonne,24,"(gold_usd_per_oz, platinum_usd_per_oz, gold_us...",1.093896,-0.205828,True,12.031107,13.356130
2,2,usd_zar,23,"(gold_usd_per_oz, platinum_usd_per_oz, gold_us...",1.010033,-0.083863,True,9.482051,22.838554
3,3,broad_usd_index,22,"(gold_usd_per_oz, platinum_usd_per_oz, gold_us...",0.759964,-0.250070,True,10.292205,33.131059
4,4,commodities,21,"(gold_usd_per_oz, platinum_usd_per_oz, gold_us...",0.728473,-0.031491,True,11.517055,44.648596
5,5,us_5y_yield,20,"(gold_usd_per_oz, platinum_usd_per_oz, gold_us...",0.695002,-0.033471,True,9.810023,54.458949
6,6,usd_zar_1w_return,19,"(gold_usd_per_oz, platinum_usd_per_oz, gold_us...",0.710203,0.015202,False,8.840080,63.299475
7,7,sa_cpi,18,"(gold_usd_per_oz, platinum_usd_per_oz, gold_us...",0.687256,-0.022948,True,9.243672,72.543566
8,8,us_fed_funds,17,"(gold_usd_per_oz, platinum_usd_per_oz, gold_us...",0.793118,0.105862,False,7.770943,80.315107
9,9,sa_real_gdp,16,"(gold_usd_per_oz, platinum_usd_per_oz, gold_us...",0.766678,-0.026440,False,8.313564,88.628993


Final regularized model using the best subset

The elimination path includes the zero-feature training-mean baseline. If that baseline wins, the final evaluation uses `DummyRegressor` rather than trying to fit an MLP to an empty feature matrix.

In [11]:
from sklearn.dummy import DummyRegressor


if best_features:
    regularized_mlp_model = make_pipeline(
        StandardScaler(),
        MLPRegressor(
            hidden_layer_sizes=(128, 64, 32),
            activation="relu",
            solver="adam",
            alpha=0.01,
            learning_rate="adaptive",
            learning_rate_init=0.001,
            max_iter=1000,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=30,
            random_state=42,
        ),
    )
    final_X = X_engineered[best_features]
else:
    print("The zero-feature baseline won; evaluating a training-mean model.")
    regularized_mlp_model = DummyRegressor(strategy="mean")
    final_X = pd.DataFrame({"constant": np.ones(len(X_engineered))}, index=X_engineered.index)

fit_cv_timeseries_model(regularized_mlp_model, final_X, y)

Fold 0, rmse: 0.992796939166568
Fold 1, rmse: 0.8091066706876633
Fold 2, rmse: 0.9492795325615034
Fold 3, rmse: 1.5974360052990728
Fold 4, rmse: 1.423687111375395
RMSEs over all splits: [0.992796939166568, 0.8091066706876633, 0.9492795325615034, 1.5974360052990728, 1.423687111375395]
Mean RMSE over all folds: 1.1544612518180406


np.float64(1.1544612518180406)